### Import Data

In [7]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.feature_selection import VarianceThreshold

print(os.getcwd())
df = pd.read_csv("../dataset/fab.csv")


d:\CODE\MyNote\SemiconductorData\code


### Data Information

In [ ]:
df.info()

# dtypes: float64(590) -> Sensor, int64(1) -> Pass_Fail, str(1) -> SensorTime
df.dtypes 

df.describe()

df.head()
df.tail()

### Set Target / Feature

In [3]:
target_col = "Pass_Fail"

feature_cols = [
    col for col in df.columns 
    if col not in ['Pass_Fail', 'SensorTime']
]

### Data Visualization

- 정상/불량 비율 확인

In [ ]:
print(df['Pass_Fail'].value_counts())

plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] =False

plt.figure(figsize=(5,5))
sns.countplot(x='Pass_Fail', data=df)
plt.title('정상 / 불량')
plt.show()

- 센서별 Pass / Fail 데이터 분포

In [ ]:
palette = {
    -1: "#4C72B0",   # Pass
     1: "#DD8452"    # Fail
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, sensor in enumerate(feature_cols[0:4]):
    sns.histplot(data=df,
                x=sensor,
                hue="Pass_Fail",
                bins=50,
                kde=True,
                palette=palette,
                alpha=0.45,
                edgecolor="black",
                linewidth=0.4,
                multiple="layer",
                ax=axes[i])
    sns.rugplot(data=df,
                x=sensor,
                hue="Pass_Fail",
                palette=palette,
                height=0.08,
                alpha=0.7,
                linewidth=1.2,
                ax=axes[i],
                legend=False)

    axes[i].set_title(f"{sensor} 데이터 분포")
    axes[i].set_xlabel("실제 측정값")
    axes[i].set_ylabel("데이터 수")

### Data Preprocessing

- 결측치 처리 

In [4]:
# 결측값이 높은 feature 제거
missing_ratio = df[feature_cols].isnull().mean().sort_values(ascending=False) * 100
missing_ratio
high_missing_features = missing_ratio[missing_ratio > 70 ].index.tolist()
df = df.drop(columns=high_missing_features)

feature_cols = [
    col for col in df.columns 
    if col not in ['Pass_Fail', 'SensorTime']
]

# 나머지 결측치 데이터 중앙값으로 처리
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())
print(df.isna().sum().sum())
print(df[feature_cols].isna().mean().sort_values(ascending=False).head())

0
Sensor0    0.0
Sensor1    0.0
Sensor2    0.0
Sensor3    0.0
Sensor4    0.0
dtype: float64


- 중복 데이터 처리

In [5]:
duplicate_count = df.duplicated(subset=feature_cols + [target_col]).sum()
duplicate_count
# 중복 데이터 확인 결과, 전체 행 기준 중복 데이터는 존재하지 않았다.

np.int64(0)

- 이상치 처리

In [ ]:
outliers_result = {}
for col in feature_cols:
    Q1 = np.nanquantile(df[col], 0.25)
    Q3 = np.nanquantile(df[col], 0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
    outliers_result[col] = len(outliers)
outliers_result = pd.Series(outliers_result).sort_values(ascending=False)
top_features = outliers_result.head(10).index
for col in top_features:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x='Pass_Fail', y=col)
    plt.title(f'{col} by Pass_Fail')
    plt.show()

# outlier_mask = (df[feature_cols] < lower_bound) | (df[feature_cols] > upper_bound)
# df["outlier_count"] = outlier_mask.sum(axis=1)
# df.groupby("Pass_Fail")["outlier_count"].describe()

### Zero Variance

In [ ]:
thresholder = VarianceThreshold(threshold=0)
_ = thresholder.fit_transform(df[feature_cols])

mask = ~thresholder.get_support()
cols_zero_var_drop = np.asarray(feature_cols)[mask].tolist()
print(f'** {len(cols_zero_var_drop)} Features to Drop by Low Variance')
print(f'{cols_zero_var_drop}')

df = df.drop(columns=cols_zero_var_drop)
feature_cols = [
    col for col in df.columns 
    if col not in ['Pass_Fail', 'SensorTime']
]

** 116 Features to Drop by Low Variance
['Sensor5', 'Sensor13', 'Sensor42', 'Sensor49', 'Sensor52', 'Sensor69', 'Sensor97', 'Sensor141', 'Sensor149', 'Sensor178', 'Sensor179', 'Sensor186', 'Sensor189', 'Sensor190', 'Sensor191', 'Sensor192', 'Sensor193', 'Sensor194', 'Sensor226', 'Sensor229', 'Sensor230', 'Sensor231', 'Sensor232', 'Sensor233', 'Sensor234', 'Sensor235', 'Sensor236', 'Sensor237', 'Sensor240', 'Sensor241', 'Sensor242', 'Sensor243', 'Sensor256', 'Sensor257', 'Sensor258', 'Sensor259', 'Sensor260', 'Sensor261', 'Sensor262', 'Sensor263', 'Sensor264', 'Sensor265', 'Sensor266', 'Sensor276', 'Sensor284', 'Sensor313', 'Sensor314', 'Sensor315', 'Sensor322', 'Sensor325', 'Sensor326', 'Sensor327', 'Sensor328', 'Sensor329', 'Sensor330', 'Sensor364', 'Sensor369', 'Sensor370', 'Sensor371', 'Sensor372', 'Sensor373', 'Sensor374', 'Sensor375', 'Sensor378', 'Sensor379', 'Sensor380', 'Sensor381', 'Sensor394', 'Sensor395', 'Sensor396', 'Sensor397', 'Sensor398', 'Sensor399', 'Sensor400', 'Sens

474